```
module purge
module avail slurm
module load xx

module avail cuda
module load cudaxx
echo $CUDA_HOME
nvcc --version
```

### concepts

- head node vs. compute node
    - “文件共享，环境复制，但硬件隔离”。
    - Login Node 和 Compute Node 就像两台不同的电脑，但它们挂载了同一个硬盘（通常是 NFS, Lustre 或 GPFS）。
        - 这意味着：你在 Login Node 的 /home/user/code 下写的 train.py，在 Compute Node 的 /home/user/code 下是完全可见且内容一致的。
    - 当你执行 srun 时，Slurm 默认不仅仅是分配机器，在大多数配置下，你在 Login Node 激活了什么 Python 环境，srun 就会把它带到 Compute Node 去。

### 运行中资源查看

- squeue：查看队列
    - squeue -u $USER
    - `squeue -t PD`: 排队中的作业
    - `squeue -t R`：running 中的作业
- `scontrol show job $JOBID`
- `seff $JOBID`: 查看 cpu 使用信息
- `sacct -j $JOBID -o JobID,AllocTRES%50`: 查看 gpu 使用信息
- alias
    - `alias sf='sinfo --Format '\''NodeList:|,NodeAddr:|,StateLong:|,Partition:|,Time:|,CPUs:|,CPUsState:|,CPUsLoad:|,FreeMem:|,Gres:|,GresUsed'\'' | column -t -s '\''|'\'''`

- 交互式（interactive）申请与执行：主要用于调试
    - salloc 申请资源：`salloc --partition=x --qos x --account=x -c 2 --mem=768G --gres=gpu:4 --time=72:00:00 -N 1`
        - `-c`: `--cpus-per-task`
    - srun -J $JOB_ID --pty /bin/bash
    - srun 将命令“发送”到计算节点执行：`srun python train.py`
        - srun 会读取 salloc 创建的环境变量（即你已经包场的 ID），把 python 命令远程传送到你刚才申请的那台高性能计算节点（DGX/HGX）上去执行。